In [1]:
# A) Preprocessing (video → frames + wav)

import os, csv, subprocess, json
from pathlib import Path

def run(cmd):
    subprocess.run(cmd, check=True)

def preprocess_video_dataset(csv_path, out_root, fps=5, sr=16000):
    """
    Reads csv with columns: emotion, filepath, split
    For each video file, extracts:
      - frames at fps into out_root/frames/<id>/
      - audio wav 16k mono into out_root/audio/<id>.wav
    Writes a manifest jsonl with resolved paths.
    """
    out_root = Path(out_root)
    (out_root/"frames").mkdir(parents=True, exist_ok=True)
    (out_root/"audio").mkdir(parents=True, exist_ok=True)
    manifest_path = out_root/"manifest.jsonl"

    with open(csv_path, "r") as f, open(manifest_path, "w") as mf:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            video_path = row["filepath"]
            emo = row["label"]
            split = row["split"]
            sample_id = f"{Path(video_path).stem}_{i}"

            frames_dir = out_root/"frames"/sample_id
            audio_path = out_root/"audio"/f"{sample_id}.wav"
            frames_dir.mkdir(parents=True, exist_ok=True)

            # Extract frames
            # -vf fps=..., scale keeps original; we'll resize in dataloader
            run([
                "ffmpeg", "-y", "-i", video_path,
                "-vf", f"fps={fps}",
                str(frames_dir/"%05d.jpg")
            ])

            # Extract audio (mono, 16k)
            run([
                "ffmpeg", "-y", "-i", video_path,
                "-ac", "1", "-ar", str(sr),
                str(audio_path)
            ])

            record = {
                "id": sample_id,
                "emotion": emo,
                "split": split,
                "frames_dir": str(frames_dir),
                "audio_wav": str(audio_path),
                "src_video": video_path
            }
            mf.write(json.dumps(record) + "\n")

    print("Wrote manifest:", manifest_path)

# preprocess_video_dataset(os.path.join('datasets', 'crema-d', 'updated_labels.csv'), os.path.join('datasets', 'crema-d-prep'), fps=5)
# preprocess_video_dataset(os.path.join('datasets', 'ravdess', 'updated_labels.csv'), os.path.join('datasets', 'ravdess-prep'), fps=5)

In [2]:
# B) Datasets (AffectNet image-only, AV manifests)

import json, random, math
import torch
import torchaudio
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

EMO_MAP = {}  # filled after scanning labels, or define fixed mapping

def build_label_map(csv_or_manifest_paths):
    labels = set()
    for p in csv_or_manifest_paths:
        p = str(p)
        if p.endswith(".csv"):
            import csv
            with open(p,"r") as f:
                r = csv.DictReader(f)
                for row in r: labels.add(row["label"])
        else:
            with open(p,"r") as f:
                for line in f:
                    labels.add(json.loads(line)["emotion"])
    labels = sorted(list(labels))
    return {lab:i for i,lab in enumerate(labels)}

class AffectNetDataset(Dataset):
    """
    CSV columns: emotion, pth, split
    images are 96x96, we upscale for pretrained backbones.
    """
    def __init__(self, csv_path, split, label_map):
        import csv
        self.items = []
        with open(csv_path, "r") as f:
            r = csv.DictReader(f)
            for row in r:
                if row["split"] == split:
                    self.items.append(row)
        self.label_map = label_map

        self.tf = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2,0.2,0.2,0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        row = self.items[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        x = self.tf(img)
        y = self.label_map[row["label"]]
        return {"image": x, "label": y}

class AVManifestDataset(Dataset):
    """
    manifest.jsonl created by preprocessing.
    Loads:
      - K frames from frames_dir
      - audio wav
    """
    def __init__(self, manifest_jsonl, split, label_map, num_frames=8, sr=16000, seconds=4.0):
        self.items = []
        with open(manifest_jsonl, "r") as f:
            for line in f:
                rec = json.loads(line)
                if rec["split"] == split:
                    self.items.append(rec)

        self.label_map = label_map
        self.num_frames = num_frames
        self.sr = sr
        self.max_len = int(sr * seconds)

        self.tf_img = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2,0.2,0.2,0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])

    def _load_frames(self, frames_dir):
        frames = sorted([str(p) for p in Path(frames_dir).glob("*.jpg")])
        if len(frames) == 0:
            # fallback: a dummy tensor
            return torch.zeros(self.num_frames, 3, 224, 224)

        # sample uniformly
        idxs = torch.linspace(0, len(frames)-1, steps=self.num_frames).long().tolist()
        out = []
        for i in idxs:
            img = Image.open(frames[i]).convert("RGB")
            out.append(self.tf_img(img))
        return torch.stack(out, dim=0)  # [T,3,224,224]

    def _load_audio(self, wav_path):
        wav, sr = torchaudio.load(wav_path)  # [1, N] usually
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        wav = wav.mean(dim=0)  # [N]
        # crop/pad to max_len
        if wav.numel() >= self.max_len:
            start = random.randint(0, wav.numel() - self.max_len)
            wav = wav[start:start+self.max_len]
        else:
            pad = self.max_len - wav.numel()
            wav = torch.nn.functional.pad(wav, (0,pad))
        return wav  # [max_len]

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        rec = self.items[idx]
        frames = self._load_frames(rec["frames_dir"])
        audio = self._load_audio(rec["audio_wav"])
        y = self.label_map[rec["emotion"]]
        return {"frames": frames, "audio": audio, "label": y, "id": rec["id"]}


In [3]:
# C) Heterogeneity augmentations (novel evaluation + training)
import torch.nn.functional as F

import random
import torch

def corrupt_frames(frames, p=0.5):
    """
    frames: [T,3,H,W] or [B,T,3,H,W]
    Applies the SAME occlusion block to all frames (and all batch items).
    """
    if random.random() > p:
        return frames

    if frames.dim() == 4:
        T, C, H, W = frames.shape
        B = None
    elif frames.dim() == 5:
        B, T, C, H, W = frames.shape
    else:
        raise ValueError(f"frames must be 4D or 5D, got {frames.shape}")

    x0 = random.randint(0, W // 2)
    y0 = random.randint(0, H // 2)
    w  = random.randint(W // 8, W // 3)
    h  = random.randint(H // 8, H // 3)

    out = frames.clone()
    if frames.dim() == 4:
        out[:, :, y0:y0+h, x0:x0+w] = 0.0
    else:
        out[:, :, :, y0:y0+h, x0:x0+w] = 0.0

    return out


def corrupt_audio(wav, p=0.5, noise_std=0.02):
    # wav: [N]
    if random.random() > p:
        return wav
    noise = torch.randn_like(wav) * noise_std
    return torch.clamp(wav + noise, -1.0, 1.0)

def modality_dropout(frames, wav, p_drop=0.15):
    # drop one modality entirely
    r = random.random()
    if r < p_drop/2:
        frames = torch.zeros_like(frames)
    elif r < p_drop:
        wav = torch.zeros_like(wav)
    return frames, wav

def temporal_misalign(wav, max_shift=0.5, sr=16000, p=0.2):
    """
    wav: [N] or [B, N]
    randomly time-shifts waveform by up to ±max_shift seconds
    """
    if random.random() > p:
        return wav

    shift = int(random.uniform(-max_shift, max_shift) * sr)
    if shift == 0:
        return wav

    if wav.dim() == 1:
        # [N]
        if shift > 0:
            return torch.cat([torch.zeros(shift, device=wav.device), wav[:-shift]], dim=0)
        else:
            s = -shift
            return torch.cat([wav[s:], torch.zeros(s, device=wav.device)], dim=0)

    elif wav.dim() == 2:
        # [B, N]
        B, N = wav.shape
        if shift > 0:
            pad = torch.zeros(B, shift, device=wav.device, dtype=wav.dtype)
            return torch.cat([pad, wav[:, :-shift]], dim=1)
        else:
            s = -shift
            pad = torch.zeros(B, s, device=wav.device, dtype=wav.dtype)
            return torch.cat([wav[:, s:], pad], dim=1)

    else:
        raise ValueError(f"Expected wav dim 1 or 2, got {wav.dim()}")


In [4]:
@torch.no_grad()
def compute_classification_metrics(y_true, y_pred, num_classes):
    """
    y_true, y_pred: 1D torch tensors (CPU or GPU)
    Returns dict with:
      accuracy, macro_f1, macro_precision, macro_recall,
      per_class_precision/recall/f1, confusion_matrix
    """
    y_true = y_true.view(-1).to(torch.long)
    y_pred = y_pred.view(-1).to(torch.long)

    # Confusion matrix: rows=true, cols=pred
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long, device=y_true.device)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1

    tp = cm.diag().to(torch.float32)
    fp = cm.sum(dim=0).to(torch.float32) - tp
    fn = cm.sum(dim=1).to(torch.float32) - tp
    tn = cm.sum().to(torch.float32) - (tp + fp + fn)

    eps = 1e-8
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)

    accuracy = tp.sum() / (cm.sum().to(torch.float32) + eps)

    macro_precision = precision.mean()
    macro_recall    = recall.mean()
    macro_f1        = f1.mean()

    return {
        "accuracy": float(accuracy.item()),
        "macro_f1": float(macro_f1.item()),
        "macro_precision": float(macro_precision.item()),
        "macro_recall": float(macro_recall.item()),
        "per_class_precision": precision.detach().cpu().tolist(),
        "per_class_recall": recall.detach().cpu().tolist(),
        "per_class_f1": f1.detach().cpu().tolist(),
        "confusion_matrix": cm.detach().cpu(),  # tensor
    }

In [5]:
# D) Model (pretrained encoders + reliability fusion

import torch
import torch.nn as nn
import torchvision

class VisionBackbone(nn.Module):
    def __init__(self, out_dim=256, train_backbone=False):
        super().__init__()
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(m.children())[:-1])  # pool output [B,512,1,1]
        self.proj = nn.Linear(512, out_dim)
        self.train_backbone = train_backbone
        # Freeze everything first
        for p in self.backbone.parameters():
            p.requires_grad = False
        
        # Unfreeze only the last ResNet block (layer4)
        for name, p in self.backbone.named_parameters():
            if "7" in name:   # in Sequential(children[:-1]), layer4 is typically index 7
                p.requires_grad = True

        # Why "7"? In ResNet18, children() indices are usually:
        # 0 conv1,1 bn1,2 relu,3 maxpool,4 layer1,5 layer2,6 layer3,7 layer4,8 avgpool,9 fc
        # Since we use children()[:-1], layer4 becomes index 7.

    def forward(self, frames):
        # frames: [B,T,3,224,224]
        B,T,C,H,W = frames.shape
        x = frames.view(B*T, C, H, W)
        z = self.backbone(x).flatten(1)  # [B*T,512]
        z = self.proj(z)                 # [B*T,out_dim]
        z = z.view(B, T, -1).mean(dim=1) # temporal average [B,out_dim]
        return z

class AudioBackbone(nn.Module):
    def __init__(self, out_dim=256, train_backbone=False):
        super().__init__()
        bundle = torchaudio.pipelines.WAV2VEC2_BASE
        self.model = bundle.get_model()
        self.train_backbone = train_backbone

        # Freeze all wav2vec2
        for p in self.model.parameters():
            p.requires_grad = False
        
        # Unfreeze only the last transformer block
        # (wav2vec2 transformer blocks are typically in self.model.encoder.transformer.layers)
        try:
            for p in self.model.encoder.transformer.layers[-1].parameters():
                p.requires_grad = True
        except Exception as e:
            print("Could not unfreeze last audio layer automatically:", e)


        # Infer feature dimension with dummy forward
        with torch.no_grad():
            dummy = torch.zeros(1, 16000)  # 1 second of audio
            feats, _ = self.model.extract_features(dummy)
            feat_dim = feats[-1].shape[-1]

        self.proj = nn.Linear(feat_dim, out_dim)

    def forward(self, wav):
        # wav: [B, N]
        feats, _ = self.model.extract_features(wav)
        h = feats[-1]            # [B, time, feat]
        h = h.mean(dim=1)        # [B, feat]
        z = self.proj(h)         # [B, out_dim]
        return z

class RAMER(nn.Module):
    def __init__(self, n_classes, z_dim=256):
        super().__init__()
        self.vision = VisionBackbone(out_dim=z_dim, train_backbone=False)
        self.audio  = AudioBackbone(out_dim=z_dim, train_backbone=False)

        self.head_v = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))
        self.head_a = nn.Sequential(nn.LayerNorm(z_dim), nn.Linear(z_dim, n_classes))

        # reliability MLP: input [z_v, z_a, disagreement(4)]
        self.rel = nn.Sequential(
            nn.Linear(z_dim*2 + 4, 256),
            nn.ReLU(),
            nn.Linear(256, 2)  # r_v, r_a
        )

    @staticmethod
    def _disagreement(z_v, z_a, logits_v, logits_a):
        p_v = torch.softmax(logits_v, dim=-1)
        p_a = torch.softmax(logits_a, dim=-1)
        absdiff = torch.abs(p_v - p_a).mean(dim=-1, keepdim=True)  # [B,1]
        kl_va = (p_v * (p_v.clamp_min(1e-8).log() - p_a.clamp_min(1e-8).log())).sum(dim=-1, keepdim=True)
        kl_av = (p_a * (p_a.clamp_min(1e-8).log() - p_v.clamp_min(1e-8).log())).sum(dim=-1, keepdim=True)
        cos   = nn.functional.cosine_similarity(z_v, z_a, dim=-1).unsqueeze(-1)  # [B,1]
        d = torch.cat([absdiff, kl_va, kl_av, cos], dim=-1)  # [B,4]
        return d

    def forward(self, frames, wav):
        z_v = self.vision(frames)
        z_a = self.audio(wav)

        logits_v = self.head_v(z_v)
        logits_a = self.head_a(z_a)

        d = self._disagreement(z_v, z_a, logits_v, logits_a)
        r_logits = self.rel(torch.cat([z_v, z_a, d], dim=-1))
        r = torch.softmax(r_logits, dim=-1)  # [B,2]
        r_v, r_a = r[:,0:1], r[:,1:2]

        logits = r_v * logits_v + r_a * logits_a
        return {
            "logits": logits,
            "logits_v": logits_v,
            "logits_a": logits_a,
            "r": r,
            "d": d
        }

In [6]:
# E) Training steps (full loop with heterogeneity regularizers)

from torch.utils.data import DataLoader
import torch.optim as optim

def ramer_loss(out, y, out_corr=None, margin=0.05,
              alpha=0.3, beta=0.3, gamma=0.5, delta=0.1, ent_lambda=0.01, agree_tau=0.2):
    """
    out: forward output on clean input
    out_corr: forward output on corrupted input (optional)
    """
    logits, logits_v, logits_a = out["logits"], out["logits_v"], out["logits_a"]
    r = out["r"]  # [B,2]

    loss_fuse = F.cross_entropy(logits, y)
    loss_uni  = alpha*F.cross_entropy(logits_v, y) + beta*F.cross_entropy(logits_a, y)

    # entropy regularizer (maximize entropy => minimize negative entropy)
    ent = -(r * (r.clamp_min(1e-8).log())).sum(dim=-1).mean()
    loss_ent = -ent_lambda * ent

    loss_corr = torch.tensor(0.0, device=logits.device)
    loss_agree = torch.tensor(0.0, device=logits.device)

    # agreement condition on clean
    p_v = torch.softmax(logits_v, dim=-1)
    p_a = torch.softmax(logits_a, dim=-1)
    D = (p_v * (p_v.clamp_min(1e-8).log() - p_a.clamp_min(1e-8).log())).sum(dim=-1) + \
        (p_a * (p_a.clamp_min(1e-8).log() - p_v.clamp_min(1e-8).log())).sum(dim=-1)
    agree_mask = (D < agree_tau).float()
    # keep reliabilities near 0.5 when modalities agree
    loss_agree = (agree_mask * ((r[:,0]-0.5)**2 + (r[:,1]-0.5)**2)).mean() * delta

    # corruption-driven hinge: reliability should drop for corrupted modality
    if out_corr is not None:
        r_clean = out["r"].detach()
        r_corr  = out_corr["r"]
        # We don't know which modality got corrupted here; caller can set flags.
        # We'll assume caller provides two separate passes (corrV and corrA).
        # So this function expects out_corr for one corruption at a time.
        # We'll compute both terms using a heuristic:
        # if corruption was visual: penalize r_v increasing
        # if corruption was audio: penalize r_a increasing
        pass

    return loss_fuse + loss_uni + loss_ent + loss_agree

def train_ramer(model, dl_train, dl_val, device="cuda",
                epochs=5, lr=3e-4,
                save_path="ramer_best.pt"):

    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_f1 = -1.0
    for ep in range(1, epochs+1):
        model.train()
        total = 0.0

        for batch in dl_train:
            frames = batch["frames"].to(device)
            wav    = batch["audio"].to(device)
            y      = batch["label"].to(device)

            frames2 = corrupt_frames(frames, p=0.25)
            wav2    = corrupt_audio(wav, p=0.25)
            frames2, wav2 = modality_dropout(frames2, wav2, p_drop=0.15)
            wav2    = temporal_misalign(wav2, p=0.1)

            out = model(frames2, wav2)

            out_clean = model(frames, wav)
            out_corrV = model(corrupt_frames(frames, p=1.0), wav)
            out_corrA = model(frames, corrupt_audio(wav, p=1.0))

            loss = ramer_loss(out, y)

            r0 = out_clean["r"].detach()
            rV = out_corrV["r"]
            rA = out_corrA["r"]
            loss_corrV = torch.relu(rV[:,0] - r0[:,0] + 0.05).mean()
            loss_corrA = torch.relu(rA[:,1] - r0[:,1] + 0.05).mean()
            loss = loss + 0.5*(loss_corrV + loss_corrA)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, 3.0)
            opt.step()

            total += loss.item()

        metrics = eval_ramer_metrics(model, dl_val, num_classes=model.head_v[-1].out_features, device=device)
        print(
            f"Epoch {ep} | train_loss={total/len(dl_train):.4f} | "
            f"val_acc={metrics['accuracy']:.4f} | val_macroF1={metrics['macro_f1']:.4f}"
        )
        best_f1 = max(best_f1, metrics["macro_f1"])

        # save best f1-macro
        # if val_acc > best_f1:
        #     best = val_acc
        #     torch.save({
        #         "model_state": model.state_dict(),
        #         "val_acc": val_acc
        #     }, save_path)
        #     print(f"Saved best RAMER model to {save_path}")

        # save best f1-macro
        if metrics["macro_f1"] >= best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": metrics['accuracy'],
                "val_macro": best_f1
            }, save_path)
            print(f"Saved best RAMER model to {save_path}")
            
    return best_f1

@torch.no_grad()
def eval_ramer_metrics(model, dl, num_classes, device="cuda"):
    model.eval()
    all_y = []
    all_p = []

    for b in dl:
        frames = b["frames"].to(device)
        wav    = b["audio"].to(device)
        y      = b["label"].to(device)

        out = model(frames, wav)
        pred = out["logits"].argmax(dim=-1)

        all_y.append(y.detach())
        all_p.append(pred.detach())

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)
    return compute_classification_metrics(y_true, y_pred, num_classes)

In [7]:
# F) Visual pretraining on AffectNet (optional but recommended)

import torchvision.models as models

class AffectNetClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        self.m = m
    def forward(self, x): return self.m(x)

def train_affectnet(model, dl_train, dl_val, device="cuda", epochs=3, lr=3e-4,
                   save_path="affectnet_best.pt"):
    model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best = 0.0

    for ep in range(1, epochs+1):
        model.train()
        for b in dl_train:
            x = b["image"].to(device)
            y = b["label"].to(device)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()

        # validation
        model.eval()
        c, t = 0, 0
        with torch.no_grad():
            for b in dl_val:
                x = b["image"].to(device)
                y = b["label"].to(device)
                pred = model(x).argmax(dim=-1)
                c += (pred == y).sum().item()
                t += y.numel()
        acc = c / max(t, 1)
        print("AffectNet epoch", ep, "val_acc", acc)

        # save best
        if acc > best:
            best = acc
            torch.save({
                "model_state": model.state_dict(),
                "val_acc": acc
            }, save_path)
            print(f"Saved best AffectNet model to {save_path}")

    return best

In [8]:
# Step 1: Build unified label map
label_map = build_label_map([
    os.path.join('datasets', 'affectNet', 'updated_labels.csv'),
    os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'),
    # "/content/prep/ravdess/manifest.jsonl",
])
print(label_map)


{'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4}


In [9]:
# Step 2: DataLoaders

from torch.utils.data import DataLoader

# AffectNet
# ds_aff_tr = AffectNetDataset(os.path.join('datasets', 'affectNet', 'updated_labels.csv'), "train", label_map)
# ds_aff_va = AffectNetDataset(os.path.join('datasets', 'affectNet', 'updated_labels.csv'), "val", label_map)

# dl_aff_tr = DataLoader(ds_aff_tr, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
# dl_aff_va = DataLoader(ds_aff_va, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

# CREMA-D AV
# ds_cre_tr = AVManifestDataset(os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'), "train", label_map, num_frames=12)
# ds_cre_va = AVManifestDataset(os.path.join('datasets', 'crema-d-prep', 'manifest.jsonl'), "val", label_map, num_frames=12)
# dl_cre_tr = DataLoader(ds_cre_tr, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
# dl_cre_va = DataLoader(ds_cre_va, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

# RAVDESS AV
ds_rav_te = AVManifestDataset(os.path.join('datasets', 'ravdess-prep', 'manifest.jsonl'), "test", label_map, num_frames=12)
dl_rav_te = DataLoader(ds_rav_te, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)


In [11]:
# Step 3: Train AffectNet visual classifier (optional but strong)

# n_classes = len(label_map)
# aff_model = AffectNetClassifier(n_classes)
# train_affectnet(aff_model, dl_aff_tr, dl_aff_va, epochs=10)

In [12]:
# Load AffectNet model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
aff_model = AffectNetClassifier(n_classes)
ckpt = torch.load("affectnet_best.pt", map_location="cpu")
aff_model.load_state_dict(ckpt["model_state"])
aff_model.to(device).eval()

AffectNetClassifier(
  (m): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [13]:
# # Step 4: Initialize RAMER and (optionally) load visual weights

# ramer = RAMER(n_classes=n_classes, z_dim=256)

# # optional: transfer resnet weights
# ramer.vision.backbone.load_state_dict(nn.Sequential(*list(aff_model.m.children())[:-1]).state_dict(), strict=True)

In [14]:
# # Step 5: Train RAMER on CREMA-D (A/V) with heterogeneity
# best_val = train_ramer(ramer, dl_cre_tr, dl_cre_va, epochs=10, lr=3e-4)

In [15]:
# Load RAMER model
ramer = RAMER(n_classes=n_classes)
ckpt = torch.load("ramer_best_wav2vec.pt", map_location="cpu")
ramer.load_state_dict(ckpt["model_state"])
ramer.to(device).eval()

RAMER(
  (vision): VisionBackbone(
    (backbone): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=

In [17]:
# acc_rav = eval_ramer(ramer, dl_rav_te)
acc_rav = eval_ramer_metrics(ramer, dl_rav_te, n_classes)
print("Zero-shot CREMA→RAVDESS ramer wav2vec acc:", acc_rav)

Zero-shot CREMA→RAVDESS ramer wav2vec acc: {'accuracy': 0.32499998807907104, 'macro_f1': 0.2648846209049225, 'macro_precision': 0.7267574071884155, 'macro_recall': 0.32500001788139343, 'per_class_precision': [0.2448979616165161, 0.5555555820465088, 1.0, 1.0, 0.8333333134651184], 'per_class_recall': [1.0, 0.3125, 0.0625, 0.0416666679084301, 0.2083333283662796], 'per_class_f1': [0.39344263076782227, 0.4000000059604645, 0.11764705926179886, 0.08000000566244125, 0.3333333432674408], 'confusion_matrix': tensor([[48,  0,  0,  0,  0],
        [32, 15,  0,  0,  1],
        [43,  1,  3,  0,  1],
        [40,  6,  0,  2,  0],
        [33,  5,  0,  0, 10]])}


In [18]:
@torch.no_grad()
def stress_test_metrics(model, dl, num_classes, device="cuda"):
    model.eval()

    def eval_mode(mode):
        all_y, all_p = [], []
        for b in dl:
            frames = b["frames"].to(device)
            wav    = b["audio"].to(device)
            y      = b["label"].to(device)

            if mode == "clean":
                pass
            elif mode == "no_audio":
                wav = torch.zeros_like(wav)
            elif mode == "no_video":
                frames = torch.zeros_like(frames)
            elif mode == "audio_noise":
                wav = corrupt_audio(wav, p=1.0, noise_std=0.05)
            elif mode == "video_occlusion":
                frames = corrupt_frames(frames, p=1.0)
            elif mode == "misalign":
                wav = temporal_misalign(wav, p=1.0)

            out = model(frames, wav)
            pred = out["logits"].argmax(dim=-1)

            all_y.append(y.detach())
            all_p.append(pred.detach())

        y_true = torch.cat(all_y, dim=0)
        y_pred = torch.cat(all_p, dim=0)
        return compute_classification_metrics(y_true, y_pred, num_classes)

    results = {}
    for m in ["clean","no_audio","no_video","audio_noise","video_occlusion","misalign"]:
        results[m] = eval_mode(m)
    return results

stress = stress_test_metrics(ramer, dl_rav_te, num_classes=n_classes)
for k,v in stress.items():
    print(k, "acc", v["accuracy"], "macroF1", v["macro_f1"])

clean acc 0.3083333373069763 macroF1 0.24036896228790283
no_audio acc 0.36666667461395264 macroF1 0.33026495575904846
no_video acc 0.21666666865348816 macroF1 0.09908471256494522
audio_noise acc 0.4333333373069763 macroF1 0.4095830023288727
video_occlusion acc 0.2958333194255829 macroF1 0.22760175168514252
misalign acc 0.32083332538604736 macroF1 0.2555284798145294
